# Store Sales — BABY CARE per-family dormant threshold, on the 0.38930 submission

**This differs from the 0.38930 submission (the family-gated blend) in exactly one way: BABY
CARE uses a 90-day dormant-zero threshold instead of the global 365-day one.** Everything
else — the four per-horizon buckets, the five seeds, the recursive half, the family-gated
blend weights, all three guards — is untouched.

## Why BABY CARE, and why 90 days

The shipped dormant-zero rule (365-day lookback, global) was itself found by a plateau sweep
across all 33 families pooled together — 330-547 days all give the same gain, and tightening
to 300 days globally *reverses the sign*, because a false positive (zeroing a row that sells a
little) costs roughly 7x what a true positive (correctly zeroing a dead line) saves.

That sweep never asked whether the tradeoff is the same for every family. Tested here with the
same leave-one-out discipline that validated the family-gated blend weight: for each of the 33
families, compare a shortened threshold (90 or 180 days) against the global 365 on **three
independent windows** (2015 backtest, 2016 backtest, real 31Jul-15Aug holdout), pick a
family's threshold from any two windows, score it blind on the third.

**Only BABY CARE passes on all three held-out folds without a single sign reversal:**

| held-out window | chosen threshold (from the other two) | held-out gain (own rows) |
|---|---|---|
| 2015 | 90 days | -0.00004 |
| 2016 | 90 days | **-0.00694** |
| 2017 | 90 days | -0.00157 |

Small — this affects one family's own rows (about 1/33 of the forecast) — and the pooled/
diluted effect on the full submission is very likely below this project's established noise
floor (~0.0039). **Shipped anyway as a deliberate small-effect test**, the same way the mixed
L2+Huber objective was shipped as a bound rather than a confident prediction.

## The warning this same test surfaced — why nothing else changed

Applying a shortened threshold to SCHOOL AND OFFICE SUPPLIES in the 2017 fold produced a
held-out gain of **+0.08158 — catastrophic**, despite a training-window signal (from
2015+2016) giving no hint anything was wrong. SCHOOL's back-to-school ramp means a 90-day
quiet spell on some store x family combo is not evidence of discontinuation the way it is for
a low-volume, low-intermittency family — it can be the calm right before the surge. This is
the same fragility this project already found testing per-family deep-lag gates on SCHOOL
(2026-08-24): **a per-family rule chosen from historical behaviour is only safe when what
varies by family is stable across years, not when it depends on that year's specific demand
regime.** BOOKS, HOME APPLIANCES, LADIESWEAR and LAWN AND GARDEN each beat 365 on *some* but
not all three folds — real candidates for a future pass, not adopted here for the same reason
three families (not one) were required before the blend-weight override shipped.

---


In [1]:
import gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

pd.set_option("display.width", 140)

COMP = "store-sales-time-series-forecasting"
REQUIRED = {"train.csv", "test.csv", "stores.csv", "holidays_events.csv"}

def has_data(p: Path) -> bool:
    try:
        return p.is_dir() and REQUIRED.issubset({f.name for f in p.iterdir() if f.is_file()})
    except OSError:
        return False

def find_data() -> Path:
    root = Path("/kaggle/input")
    if root.is_dir():
        for cand in [root / COMP, *sorted(d for d in root.iterdir() if d.is_dir())]:
            if has_data(cand):
                return cand
    for cand in (Path("data"), Path("../data"), Path("../../data")):
        if has_data(cand):
            return cand
    try:
        import kagglehub
        got = Path(kagglehub.competition_download(COMP))
        if has_data(got):
            return got
        for sub in got.rglob("*"):
            if has_data(sub):
                return sub
    except Exception as exc:
        print(f"kagglehub fallback failed: {exc}")
    raise FileNotFoundError(
        f"Could not find {sorted(REQUIRED)}. In the Kaggle editor: + Add Input -> "
        "Competitions -> store-sales-time-series-forecasting.")

DATA = find_data()
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("submissions/blend")
OUT.mkdir(parents=True, exist_ok=True)

HORIZON = 16
MODEL_START = pd.Timestamp("2015-01-01")
EQ_START, EQ_END = pd.Timestamp("2016-04-16"), pd.Timestamp("2016-04-22")
DORMANT_DAYS = 365
N_LAGS_REC = 63          # recursive half: target lags 1..63
BLEND_W = 0.70           # direct model's share, in log space -- see the note above

# The last validated submission's chain-wide total, as a final sanity check. A blend bug once
# shipped half this volume and scored 0.66126 because nobody read the number before uploading.
VALIDATED_VOLUME = 12_635_225      # the 0.39079 blend submission's chain-wide total

DTYPES = {"store_nbr": "int8", "family": "category", "onpromotion": "int32", "sales": "float32"}
train = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=DTYPES)
test = pd.read_csv(DATA / "test.csv", parse_dates=["date"], dtype=DTYPES)
stores = pd.read_csv(DATA / "stores.csv", dtype={"store_nbr": "int8"})
hol = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])

TRAIN_END = train.date.max()
VALID_START = TRAIN_END - pd.Timedelta(days=HORIZON - 1)
print(f"train {train.date.min():%Y-%m-%d} -> {TRAIN_END:%Y-%m-%d} | "
      f"test {test.date.min():%Y-%m-%d} -> {test.date.max():%Y-%m-%d}")

train 2013-01-01 -> 2017-08-15 | test 2017-08-16 -> 2017-08-31


---
## Shared panel: Christmas restored, earthquake repaired

Both halves train on the same cleaned history, so the blend is comparing forecasting strategies rather than data preparation.

In [2]:
t0 = time.time()
full_idx = pd.date_range(train.date.min(), test.date.max(), freq="D")

both = pd.concat([train.drop(columns="sales"), test], ignore_index=True)
both["family"] = both.family.astype(str)
sales_w = (train.assign(family=train.family.astype(str))
           .pivot(index="date", columns=["store_nbr", "family"], values="sales")
           .reindex(full_idx).sort_index(axis=1))
promo_w = (both.pivot(index="date", columns=["store_nbr", "family"], values="onpromotion")
           .reindex(full_idx).sort_index(axis=1).fillna(0.0))
xmas = pd.DatetimeIndex([d for d in full_idx
                         if d <= TRAIN_END and d not in set(train.date.unique())])
sales_w.loc[xmas] = 0.0
assert sales_w.loc[:TRAIN_END].isna().sum().sum() == 0

def repair_window(W, start, end, halo_weeks=8):
    out = W.copy(); win = pd.date_range(start, end)
    ctx = W.loc[start - pd.Timedelta(weeks=halo_weeks): end + pd.Timedelta(weeks=halo_weeks)]
    ctx = ctx.drop(index=win, errors="ignore")
    by_dow = ctx.groupby(ctx.index.dayofweek).median()
    for d in win:
        out.loc[d] = by_dow.loc[d.dayofweek].values
    return out

sales_w = repair_window(sales_w, EQ_START, EQ_END)
eq_dates = pd.date_range(EQ_START, EQ_END)
SERIES = sales_w.columns
n_s = len(SERIES)
L = np.log1p(sales_w).astype("float32")
P = np.log1p(promo_w).astype("float32")
ZERO = (sales_w == 0).astype("float32")
print(f"panel {sales_w.shape}, {n_s} series  ({time.time()-t0:.0f}s)")

panel (1704, 1782), 1782 series  (40s)


In [3]:
h = hol.copy()
h = h[~((h.type == "Holiday") & (h.transferred))]
h.loc[h.type == "Transfer", "type"] = "Holiday"
work_days = set(h.loc[h.type == "Work Day", "date"])
h = h[h.type != "Work Day"]
events = h[h.type == "Event"]; h = h[h.type != "Event"]
nat = pd.DatetimeIndex(sorted(set(h.loc[h.locale == "National", "date"])))
nat_name = (h[h.locale == "National"].drop_duplicates("date")[["date", "description"]]
            .rename(columns={"description": "nat_hol_name"}))
loc_name = (h[h.locale == "Local"].drop_duplicates(["date", "locale_name"])
            [["date", "locale_name", "description"]]
            .rename(columns={"locale_name": "city", "description": "loc_hol_name"}))
geo = stores.set_index("store_nbr")[["city", "state", "type", "cluster"]]
geo.columns = ["city", "state", "store_type", "cluster"]
assert not (set(loc_name.city) - set(stores.city))

pos = np.searchsorted(nat.values, full_idx.values)
prev_d = np.where(pos > 0, (full_idx.values - nat.values[np.maximum(pos-1, 0)])
                  / np.timedelta64(1, "D"), 999)
next_d = np.where(pos < len(nat), (nat.values[np.minimum(pos, len(nat)-1)] - full_idx.values)
                  / np.timedelta64(1, "D"), 999)
cal_hol = pd.DataFrame({"date": full_idx,
                        "work_day": full_idx.isin(work_days).astype("int8"),
                        "is_event": full_idx.isin(set(events.date)).astype("int8"),
                        "days_since_nat": np.clip(prev_d, 0, 30).astype("int16"),
                        "days_to_nat": np.clip(next_d, 0, 30).astype("int16")})
print(f"{len(nat_name)} national + {len(loc_name)} local holiday dates")

102 national + 147 local holiday dates


---
# Half 1 — the direct per-horizon model

Unchanged from the 0.39586 submission. Four models, each using the freshest lag its horizon
range legally allows.

**The legality rule, asserted rather than trusted.** A target date `d` at horizon `h` has
forecast origin `o = d − h`; feature `lag_k` is `sales(d − k)`, known at the origin iff
`d − k ≤ d − h`, i.e. **`k ≥ h`**.

In [4]:
promo_feats = {}
promo_feats["promo"] = P
promo_feats["promo_rmean_7"] = P.rolling(7, min_periods=1).mean()
promo_feats["promo_rmean_28"] = P.rolling(28, min_periods=3).mean()
promo_feats["promo_lag_16"] = P.shift(HORIZON)
promo_feats["promo_rel_112"] = P - P.rolling(112, min_periods=14).mean()
promo_feats["promo_rel_28"] = P - P.rolling(28, min_periods=5).mean()
for k in (1, 2, 3, 7):
    promo_feats[f"promo_lead_{k}"] = P.shift(-k)
promo_feats["promo_fwd7"] = P.shift(-6).rolling(7, min_periods=1).mean()
chain = P.mean(axis=1); ones = np.ones(n_s, dtype="float32")
promo_feats["promo_chain_level"] = pd.DataFrame(
    np.outer(chain.to_numpy(dtype="float32"), ones), index=full_idx, columns=SERIES)
promo_feats["promo_chain_rel"] = pd.DataFrame(
    np.outer((chain - chain.rolling(112, min_periods=14).mean()).to_numpy(dtype="float32"),
             ones), index=full_idx, columns=SERIES)
del chain; gc.collect()
praw = np.expm1(P)
fam_p = np.log1p(praw.T.groupby(level=1).sum().T)
sto_p = np.log1p(praw.T.groupby(level=0).sum().T)
def _b(a, lv): return a[SERIES.get_level_values(lv)].set_axis(SERIES, axis=1)
promo_feats["fam_promo_rel"] = _b(fam_p - fam_p.rolling(112, min_periods=14).mean(), 1)
promo_feats["fam_promo_fwd7"] = _b(fam_p.shift(-6).rolling(7, min_periods=1).mean()
                                   - fam_p.rolling(112, min_periods=14).mean(), 1)
promo_feats["store_promo_rel"] = _b(sto_p - sto_p.rolling(112, min_periods=14).mean(), 0)
del praw, fam_p, sto_p; gc.collect()

A_pos = (sales_w.to_numpy() > 0)
gap = np.empty(A_pos.shape, dtype="float32"); _last = np.full(A_pos.shape[1], -999.0)
for i in range(A_pos.shape[0]):
    gap[i] = i - _last
    _last = np.where(A_pos[i], float(i), _last)
GAP = pd.DataFrame(np.minimum(gap, 999.0), index=sales_w.index, columns=SERIES)
del gap, A_pos; gc.collect()

mask = full_idx >= MODEL_START
dates_sel = full_idx[mask]
LAG_OFFSETS = (0, 1, 2, 3, 4, 5, 6, 12, 19, 33, 47)

def build_design(min_lag):
    feats = dict(promo_feats)
    for d in LAG_OFFSETS:
        feats[f"lag_{min_lag + d}"] = L.shift(min_lag + d)
    base = L.shift(min_lag)
    for w in (7, 14, 28, 56, 112):
        feats[f"rmean_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).mean()
    for w in (14, 28):
        feats[f"rstd_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).std()
    feats["rmax_28"] = base.rolling(28, min_periods=7).max()
    dow_start = 7 * int(np.ceil(min_lag / 7))
    feats["dow_mean_4"] = sum(L.shift(dow_start + 7 * k) for k in range(4)) / 4
    feats["dow_mean_8"] = sum(L.shift(dow_start + 7 * k) for k in range(8)) / 8
    feats["zfrac_28"] = ZERO.shift(min_lag).rolling(28, min_periods=7).mean()
    feats["zfrac_112"] = ZERO.shift(min_lag).rolling(112, min_periods=28).mean()
    feats["days_since_sale"] = GAP.shift(min_lag)
    d = pd.DataFrame({
        "date": np.repeat(dates_sel.values, n_s),
        "store_nbr": np.tile(SERIES.get_level_values(0).to_numpy(), len(dates_sel)),
        "family": np.tile(SERIES.get_level_values(1).to_numpy(), len(dates_sel))})
    for nm, W in feats.items():
        d[nm] = W.to_numpy(dtype="float32")[mask].ravel()
    d["target"] = L.to_numpy(dtype="float32")[mask].ravel()
    del feats; gc.collect()
    d = d.join(geo, on="store_nbr")
    dt = d.date
    d["dow"] = dt.dt.dayofweek.astype("int8"); d["day"] = dt.dt.day.astype("int8")
    d["month"] = dt.dt.month.astype("int8"); d["year"] = dt.dt.year.astype("int16")
    d["dayofyear"] = dt.dt.dayofyear.astype("int16")
    d["is_weekend"] = (d.dow >= 5).astype("int8")
    d["days_to_month_end"] = (dt.dt.days_in_month - dt.dt.day).astype("int8")
    d["payday_window"] = (dt.dt.day.isin([15,16,17,1,2,3]) | (d.days_to_month_end <= 1)).astype("int8")
    d = d.merge(cal_hol, on="date", how="left").merge(nat_name, on="date", how="left")
    d = d.merge(loc_name, on=["date", "city"], how="left")
    d["nat_hol_name"] = d.nat_hol_name.fillna("none")
    d["loc_hol_name"] = d.loc_hol_name.fillna("none")
    for c in ("family","city","state","store_type","nat_hol_name","loc_hol_name"):
        d[c] = d[c].astype("category")
    d["store_nbr"] = d.store_nbr.astype("int16"); d["cluster"] = d.cluster.astype("int16")
    cols = [c for c in d.columns if c not in ("date", "target")]
    assert len(cols) == 60, f"expected 60 features, got {len(cols)}"
    return d, cols

CATS = ["family","city","state","store_type","nat_hol_name","loc_hol_name"]
PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.08, num_leaves=96,
              min_data_in_leaf=50, feature_fraction=0.75, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=1.0, feature_pre_filter=False,
              num_threads=0, verbose=-1, seed=42)
SEED_CFG = [(42,0.75,0.80), (79,0.60,0.90), (116,0.85,0.70), (153,0.70,0.85), (190,0.65,0.75)]
def seed_params(i):
    p = dict(PARAMS)
    p["seed"], p["feature_fraction"], p["bagging_fraction"] = SEED_CFG[i]
    return p
def rmsle(yt, yp):
    yp = np.clip(np.asarray(yp, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(yp) - np.log1p(np.asarray(yt, float))) ** 2)))

ASSEMBLY = {1:1, 2:2, 3:4, 4:4, **{h: 16 for h in range(5, 17)}}
MIN_LAGS = sorted(set(ASSEMBLY.values()))
for day, ml in ASSEMBLY.items():
    assert ml >= day, f"day {day} cannot use sales only {ml} days old"
print(f"{len(MIN_LAGS)} direct models {MIN_LAGS} covering 16 forecast days")

4 direct models [1, 2, 4, 16] covering 16 forecast days


In [5]:
val_log_direct, best_rounds = {}, {}
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    tr_m = (d.date < VALID_START) & d.target.notna() & ~eq_row
    va_m = (d.date >= VALID_START) & (d.date <= TRAIN_END)
    day_of = ((d.loc[va_m,"date"].to_numpy() - np.datetime64(VALID_START))
              / np.timedelta64(1,"D")).astype(int) + 1
    serves = np.isin(day_of, [dd for dd, ml in ASSEMBLY.items() if ml == min_lag])
    es = np.zeros(len(d), dtype=bool)
    es[np.flatnonzero(va_m.to_numpy())[np.flatnonzero(serves)]] = True
    dtr = lgb.Dataset(d.loc[tr_m, cols], d.loc[tr_m,"target"],
                      categorical_feature=CATS, free_raw_data=False)
    dva = lgb.Dataset(d.loc[es, cols], d.loc[es,"target"],
                      categorical_feature=CATS, reference=dtr, free_raw_data=False)
    m0 = lgb.train(seed_params(0), dtr, num_boost_round=2400, valid_sets=[dva],
                   callbacks=[lgb.early_stopping(150, verbose=False)])
    best_rounds[min_lag] = m0.num_trees()
    logs = [m0.predict(d.loc[va_m, cols])]
    for i in range(1, len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dtr, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[va_m, cols]))
        del mi; gc.collect()
    val_log_direct[min_lag] = np.mean(logs, axis=0)
    if min_lag == 16:
        y_va = np.expm1(d.loc[va_m,"target"].to_numpy())
        DAY_OF = day_of
    print(f"  direct lag>={min_lag:>2}: {best_rounds[min_lag]:>5} trees ({time.time()-t:.0f}s)",
          flush=True)
    del d, dtr, dva, m0, logs; gc.collect()

direct_val = np.empty(len(DAY_OF))
for day, ml in ASSEMBLY.items():
    sel = DAY_OF == day
    direct_val[sel] = val_log_direct[ml][sel]
print(f"\ndirect half, holdout: {rmsle(y_va, np.expm1(direct_val)):.5f}")

  direct lag>= 1:   920 trees (742s)
  direct lag>= 2:   508 trees (482s)
  direct lag>= 4:  1813 trees (1300s)
  direct lag>=16:  1388 trees (1093s)

direct half, holdout: 0.38866


---
# Half 2 — the recursive per-family model

One model per product family (33 of them), each trained across that family's 54 store-series to
predict **one day ahead** from target lags 1–63, then rolled forward sixteen times with each
prediction written back as the next day's `lag_1`.

Three choices reproduced from the public architecture because they are load-bearing there:

- **Per-family partitioning.** This project's own backtests found per-family *helps* recursive
  models and *hurts* direct ones — opposite signs, so it cannot be ported blindly in either
  direction.
- **`log1p` then per-series MinMax.** Fifty-four series share one small model, so putting them
  on a common 0–1 scale lets it fit shape rather than level. (Tested on the *direct* model,
  where it lost by 6.5 σ — the same ingredient is right here and wrong there.)
- **Stock LightGBM, 100 trees, no early stopping.** Note this makes the recursive half
  **deterministic**: the defaults do no row or column subsampling, so `random_state` has no
  effect. There is no seed ensemble on this side, and unexploited diversity remains.

Deliberately *not* copied: `oil` and `transactions` (rejected here five ways and one way
respectively, and the source notebook runs no ablation, so they are suspects rather than proven
ingredients) and the pruned 7-holiday encoding (we measured that holiday *names* win).

In [6]:
cal = pd.DataFrame(index=full_idx)
cal["dow"] = full_idx.dayofweek; cal["day"] = full_idx.day
cal["month"] = full_idx.month; cal["dayofyear"] = full_idx.dayofyear
cal["is_weekend"] = (full_idx.dayofweek >= 5).astype(int)
cal["days_to_month_end"] = full_idx.days_in_month - full_idx.day
cal["payday_window"] = (full_idx.day.isin([15,16,17,1,2,3]) | (cal.days_to_month_end <= 1)).astype(int)
cal["work_day"] = full_idx.isin(work_days).astype(int)
cal["is_event"] = full_idx.isin(set(events.date)).astype(int)
cal["days_since_nat"] = np.clip(prev_d, 0, 30)
cal["days_to_nat"] = np.clip(next_d, 0, 30)
cal["nat_hol"] = pd.Categorical(
    nat_name.set_index("date")["nat_hol_name"].reindex(full_idx)).codes
cal_np = cal.to_numpy(dtype="float32")
date_pos = {d: i for i, d in enumerate(full_idx)}

geo_num = geo.copy()
for c in ("city", "state", "store_type"):
    geo_num[c] = geo_num[c].astype("category").cat.codes
FAMILIES = sorted(SERIES.get_level_values(1).unique())
eq_set = set(pd.date_range(EQ_START, EQ_END))

def run_family(fam, fit_end, target_dates):
    # Train on this family's 54 store-series up to fit_end, then roll forward over target_dates.
    cols = [c for c in SERIES if c[1] == fam]
    Lf = L[cols].to_numpy(dtype="float32")
    Pf = P[cols].to_numpy(dtype="float32")
    n_d, n_st = Lf.shape
    st_arr = np.array([c[0] for c in cols])
    stat = geo_num.loc[st_arr, ["city","state","store_type","cluster"]].to_numpy(dtype="float32")
    stat = np.column_stack([st_arr.astype("float32"), stat])

    # Per-series MinMax fitted on training data only -- never on the window being forecast.
    fit_hi = date_pos[fit_end] + 1
    lo = Lf[:fit_hi].min(axis=0); hi = Lf[:fit_hi].max(axis=0)
    rng = np.where(hi - lo > 1e-9, hi - lo, 1.0)
    S = (Lf - lo) / rng

    start = max(date_pos[MODEL_START], N_LAGS_REC)
    rows_d = np.array([di for di in range(start, fit_hi) if full_idx[di] not in eq_set])

    def make_X(dis, panel):
        n = len(dis)
        lagm = np.empty((n, n_st, N_LAGS_REC), dtype="float32")
        for k in range(1, N_LAGS_REC + 1):
            lagm[:, :, k-1] = panel[dis - k]
        lagm = lagm.reshape(n * n_st, N_LAGS_REC)
        promo = Pf[dis].reshape(-1, 1)
        leads = np.column_stack([Pf[np.minimum(dis + k, n_d - 1)].ravel() for k in (1,2,3,7)])
        return np.column_stack([lagm, promo, leads,
                                np.repeat(cal_np[dis], n_st, axis=0),
                                np.tile(stat, (n, 1))])

    X = make_X(rows_d, S); yv = S[rows_d].ravel()
    ok = np.isfinite(X).all(axis=1) & np.isfinite(yv)
    m = lgb.LGBMRegressor(n_estimators=100, random_state=0, verbose=-1, n_jobs=-1)
    m.fit(X[ok], yv[ok])

    Sw = S.copy()
    out = np.empty((len(target_dates), n_st), dtype="float32")
    for step, dd in enumerate(target_dates):
        di = date_pos[dd]
        p = m.predict(make_X(np.array([di]), Sw))
        Sw[di] = p                      # feed the prediction back as tomorrow's lag_1
        out[step] = p
    return out * rng + lo, cols         # invert the scaler, still in log space

In [7]:
va_dates = pd.date_range(VALID_START, TRAIN_END)
t = time.time()
rec_val_frame = pd.DataFrame(index=va_dates, columns=SERIES, dtype="float32")
for i, fam in enumerate(FAMILIES):
    p, cols = run_family(fam, VALID_START - pd.Timedelta(days=1), va_dates)
    rec_val_frame[cols] = p
    if (i + 1) % 11 == 0:
        print(f"  {i+1}/{len(FAMILIES)} families ({time.time()-t:.0f}s)", flush=True)
recursive_val = rec_val_frame.to_numpy().ravel()      # date-major, series-fastest
print(f"recursive half, holdout: {rmsle(y_va, np.expm1(recursive_val)):.5f}  "
      f"({time.time()-t:.0f}s)")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  11/33 families (18s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  22/33 families (37s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  33/33 families (57s)
recursive half, holdout: 0.39893  (57s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [8]:
print(f"direct    {rmsle(y_va, np.expm1(direct_val)):.5f}")
print(f"recursive {rmsle(y_va, np.expm1(recursive_val)):.5f}")
r_d = direct_val - np.log1p(y_va)
r_r = recursive_val - np.log1p(y_va)
corr = np.corrcoef(r_d, r_r)[0, 1]
print(f"residual correlation: {corr:.3f}   "
      f"(seed-to-seed within one model is 0.972-0.979 -- lower means genuinely different)")
print()
print("blend sweep (w = direct share):")
for w in (1.0, 0.9, 0.8, 0.75, 0.70, 0.65, 0.6, 0.5, 0.0):
    mark = "  <- shipping" if abs(w - BLEND_W) < 1e-9 else ""
    print(f"  w={w:.2f}  {rmsle(y_va, np.expm1(w*direct_val + (1-w)*recursive_val)):.5f}{mark}")
print()
print("per-day, to confirm the complementarity holds this run:")
rows = []
for day in (1, 2, 4, 8, 12, 16):
    sel = DAY_OF == day
    rows.append({"day": day,
                 "direct": rmsle(y_va[sel], np.expm1(direct_val[sel])),
                 "recursive": rmsle(y_va[sel], np.expm1(recursive_val[sel]))})
bd = pd.DataFrame(rows)
bd["recursive wins by"] = bd.direct - bd.recursive
print(bd.to_string(index=False, float_format=lambda v: f"{v: .5f}"))
print()
print("The recursive column must WORSEN with the forecast day -- that is compounding error, and")
print("its absence would mean the rollout is somehow seeing the future.")
assert bd.recursive.iloc[-1] > bd.recursive.iloc[0], "recursive half does not degrade with horizon"


direct    0.38866
recursive 0.39893
residual correlation: 0.917   (seed-to-seed within one model is 0.972-0.979 -- lower means genuinely different)

blend sweep (w = direct share):
  w=1.00  0.38866
  w=0.90  0.38667
  w=0.80  0.38534
  w=0.75  0.38493
  w=0.70  0.38469  <- shipping
  w=0.65  0.38461
  w=0.60  0.38471
  w=0.50  0.38542
  w=0.00  0.39893

per-day, to confirm the complementarity holds this run:
 day   direct  recursive  recursive wins by
   1  0.38005    0.38488           -0.00483
   2  0.36759    0.38063           -0.01304
   4  0.36769    0.38286           -0.01517
   8  0.39160    0.38629            0.00531
  12  0.40139    0.43411           -0.03272
  16  0.44508    0.41187            0.03321

The recursive column must WORSEN with the forecast day -- that is compounding error, and
its absence would mean the rollout is somehow seeing the future.


---
## Refit both halves on all history and write the submission

In [9]:
test_log_direct = {}
te_key = None
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    full_tr = (d.date <= TRAIN_END) & d.target.notna() & ~eq_row
    te_m = d.date > TRAIN_END
    dfull = lgb.Dataset(d.loc[full_tr, cols], d.loc[full_tr,"target"],
                        categorical_feature=CATS, free_raw_data=False)
    logs = []
    for i in range(len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dfull, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[te_m, cols]))
        del mi; gc.collect()
    test_log_direct[min_lag] = np.mean(logs, axis=0)
    if te_key is None:
        te_key = d.loc[te_m, ["date","store_nbr","family"]].copy()
    print(f"  direct lag>={min_lag:>2} refit ({time.time()-t:.0f}s)", flush=True)
    del d, dfull, logs; gc.collect()

te_day = ((te_key.date.to_numpy() - np.datetime64(TRAIN_END)) / np.timedelta64(1,"D")).astype(int)
assert te_day.min() == 1 and te_day.max() == 16
direct_test = np.empty(len(te_key))
for day, ml in ASSEMBLY.items():
    direct_test[te_day == day] = test_log_direct[ml][te_day == day]

  direct lag>= 1 refit (773s)
  direct lag>= 2 refit (481s)
  direct lag>= 4 refit (1378s)
  direct lag>=16 refit (1093s)


In [10]:
te_dates = pd.date_range(TRAIN_END + pd.Timedelta(days=1), test.date.max())
t = time.time()
rec_test_frame = pd.DataFrame(index=te_dates, columns=SERIES, dtype="float32")
for i, fam in enumerate(FAMILIES):
    p, cols = run_family(fam, TRAIN_END, te_dates)
    rec_test_frame[cols] = p
    if (i + 1) % 11 == 0:
        print(f"  {i+1}/{len(FAMILIES)} families ({time.time()-t:.0f}s)", flush=True)
recursive_test = rec_test_frame.to_numpy().ravel()
print(f"recursive refit + rollout done ({time.time()-t:.0f}s)")

# te_key is date-major with series varying fastest, which is exactly how rec_test_frame ravels.
# Assert it rather than trust it -- blending two misaligned vectors would produce a plausible
# looking number and a ruined submission.
exp_store = np.tile(SERIES.get_level_values(0).to_numpy(), len(te_dates))
exp_fam = np.tile(SERIES.get_level_values(1).to_numpy(), len(te_dates))
assert (te_key.store_nbr.to_numpy() == exp_store).all(), "row order mismatch (store)"
assert (te_key.family.astype(str).to_numpy() == exp_fam).all(), "row order mismatch (family)"
print("row alignment between the two halves verified")

print(f"mean log prediction -- direct {direct_test.mean():.3f} | "
      f"recursive {recursive_test.mean():.3f}")
assert abs(direct_test.mean() - recursive_test.mean()) < 1.0, \
    "the halves disagree on scale -- one of them did not train properly"

# Three families use a leave-one-out-validated weight instead of the uniform BLEND_W --
# see the notebook introduction for how these were chosen and why only these three.
FAMILY_WEIGHT = {"SCHOOL AND OFFICE SUPPLIES": 0.35, "HOME APPLIANCES": 0.90, "AUTOMOTIVE": 0.50}
te_fam = te_key.family.astype(str).to_numpy()
w_per_row = np.full(len(te_key), BLEND_W)
for fam, w in FAMILY_WEIGHT.items():
    w_per_row[te_fam == fam] = w
print("family-gated weights: " + ", ".join(f"{f}={w}" for f, w in FAMILY_WEIGHT.items())
      + f"; all other families keep {BLEND_W}")
print(f"rows touched: {int((w_per_row != BLEND_W).sum()):,} of {len(te_key):,}")
final_log = w_per_row * direct_test + (1 - w_per_row) * recursive_test
pred = np.clip(np.expm1(final_log), 0, None)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  11/33 families (18s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  22/33 families (37s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  33/33 families (57s)
recursive refit + rollout done (57s)
row alignment between the two halves verified
mean log prediction -- direct 3.624 | recursive 3.624
family-gated weights: SCHOOL AND OFFICE SUPPLIES=0.35, HOME APPLIANCES=0.9, AUTOMOTIVE=0.5; all other families keep 0.7
rows touched: 2,592 of 28,512


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [11]:
# Product lines with no sales in the trailing lookback are forced to exactly zero. A pooled
# model cannot represent exact zero; under RMSLE these rows are 3.6% of the forecast, 0.01%
# of the units, and were worth -0.0153 on the leaderboard at the global 365-day threshold.
#
# BABY CARE uses a shorter 90-day threshold instead of the global 365 -- leave-one-out
# validated across all 3 independent windows (2015/2016 backtest, 2017 holdout), zero sign
# reversal, mean held-out gain on its own rows ~-0.003. Every other family keeps 365 -- a
# blanket shortened threshold was NOT adopted: it reverses catastrophically on SCHOOL AND
# OFFICE SUPPLIES (+0.08 RMSLE on its own rows in the 2017 fold), whose back-to-school ramp
# means a 90-day quiet spell is not evidence of discontinuation.
FAMILY_DORMANT_DAYS = {"BABY CARE": 90}

def dead_combos(days, family_filter=None):
    cutoff = TRAIN_END - pd.Timedelta(days=days)
    recent = (train[train.date > cutoff]
              .groupby(["store_nbr","family"], observed=True).sales.sum())
    d = {(s, str(f)) for (s, f), v in recent.items() if v == 0}
    all_k = {(s, str(f)) for s, f in zip(test.store_nbr, test.family.astype(str))}
    d |= all_k - {(s, str(f)) for s, f in recent.index}
    if family_filter is not None:
        d = {(s, f) for s, f in d if f == family_filter}
    return d

dead = dead_combos(DORMANT_DAYS)
for fam, days in FAMILY_DORMANT_DAYS.items():
    dead = {(s, f) for s, f in dead if f != fam}     # drop this family's global-threshold set
    dead |= dead_combos(days, family_filter=fam)      # replace with its own threshold

keys = list(zip(te_key.store_nbr.to_numpy(), te_key.family.astype(str).to_numpy()))
dormant = np.array([k in dead for k in keys])
leaked = pred[dormant]
print(f"dormant lines: {len(dead)} combinations, {dormant.sum():,} rows ({dormant.mean():.1%}), "
      f"carrying {leaked.sum():,.0f} units ({leaked.sum()/pred.sum():.3%} of volume)")
print("family overrides: " + ", ".join(f"{f}={d}d" for f, d in FAMILY_DORMANT_DAYS.items())
      + f"; all other families keep {DORMANT_DAYS}d")
pred = pred.copy(); pred[dormant] = 0.0

out = te_key.copy(); out["sales"] = pred
out["family"] = out.family.astype(str)
key = test.assign(family=test.family.astype(str))[["id","date","store_nbr","family"]]
submission = key.merge(out, on=["date","store_nbr","family"], how="left")
assert len(submission) == len(test)
assert submission.sales.notna().all()
assert (submission.sales >= 0).all()
assert submission.id.equals(test.id)

volume = submission.sales.sum(); ratio = volume / VALIDATED_VOLUME
print(f"\ntotal predicted units : {volume:,.0f}")
print(f"last validated        : {VALIDATED_VOLUME:,.0f}  (ratio {ratio:.3f})")
assert 0.85 < ratio < 1.15, f"volume is {ratio:.2f}x the validated model's -- do not submit"

daily = submission.groupby("date").sales.sum()
print("\ndaily chain-wide forecast volume:")
print(daily.to_string(float_format=lambda v: f"{v:,.0f}"))
print(f"\nlargest day-over-day change: {daily.pct_change().abs().max():.1%} "
      "(the weekly cycle alone moves sales about this much)")
print("Check continuity before submitting -- the weekly rhythm should be the only visible pattern.")

submission[["id","sales"]].to_csv(OUT / "submission.csv", index=False)
print(f"\nwritten -> {(OUT / 'submission.csv').resolve()}")
print(f"blend weight: {BLEND_W} direct / {1-BLEND_W:.2f} recursive; BABY CARE dormant threshold: 90d")


dormant lines: 73 combinations, 1,168 rows (4.1%), carrying 423 units (0.003% of volume)
family overrides: BABY CARE=90d; all other families keep 365d

total predicted units : 12,633,718
last validated        : 12,635,225  (ratio 1.000)

daily chain-wide forecast volume:
date
2017-08-16     813,347
2017-08-17     645,170
2017-08-18     765,592
2017-08-19     894,115
2017-08-20   1,022,349
2017-08-21     798,366
2017-08-22     725,468
2017-08-23     760,425
2017-08-24     638,483
2017-08-25     752,844
2017-08-26     905,618
2017-08-27     998,687
2017-08-28     758,777
2017-08-29     696,850
2017-08-30     766,807
2017-08-31     690,817

largest day-over-day change: 24.0% (the weekly cycle alone moves sales about this much)
Check continuity before submitting -- the weekly rhythm should be the only visible pattern.

written -> /kaggle/working/submission.csv
blend weight: 0.7 direct / 0.30 recursive; BABY CARE dormant threshold: 90d


---
## Reading the result

| Reference | Leaderboard |
|---|---|
| + per-horizon models | 0.41113 |
| + dormant-series zero | 0.39586 |
| + recursive blend | 0.39079 |
| + mixed L2+Huber (rejected) | 0.39320 |
| + `rmean_3` near buckets (flat) | 0.39094 |
| **+ family-gated blend weight** | **0.38930 <- the number this must beat** |
| this notebook (BABY CARE 90-day dormant threshold) | ? |

**What this tests.** A single, small, leave-one-out-validated per-family refinement of the
365-day dormant-zero rule -- BABY CARE only, everything else unchanged. Mean held-out gain
across the 3 leave-one-out folds was small (-0.00004 / -0.00694 / -0.00157 on BABY CARE's own
rows), touching roughly 1/33 of the forecast, so the realistic expectation diluted into the
full submission is a small movement, plausibly inside this project's ~0.0004 Kaggle-vs-local
environment-noise band -- shipped to see whether the structural-limitation category (the same
category the original dormant-zero rule belongs to, which transferred at 5.6x) transfers here
too, however small the local signal.

**If this lands flat or worse:** treat it the way `rmean_3` was treated -- a real, correctly
out-of-sample local signal that didn't survive dilution to the full submission -- not a reason
to distrust the leave-one-out method itself, which has now produced one clear win (family-gated
blend weight, 0.78x transfer) and one ambiguous small-effect result out of two uses.
